# NFPP Sodium-Ion Cell & Solar Energy Dispatch Research Pipeline

This notebook implements the complete research pipeline for the Sodium Iron Pyrophosphate (NFPP) battery system co-optimization and Model-Informed Energy Dispatch in Hybrid Solar–Battery Energy Storage Systems.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass

# Environment Setup
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('sodium-ion-ess'):
        !git clone https://github.com/mhizterpaul/sodium-ion-ess.git
        %cd sodium-ion-ess
    sys.path.append(os.getcwd())

# MP API Key configuration
if 'MP_API_KEY' not in os.environ:
    os.environ['MP_API_KEY'] = getpass("Enter Materials Project API Key: ")

!pip install pybamm numpy scipy matplotlib requests mp-api pymatgen pymoo mpi4py pint ufl
!add-apt-repository -y ppa:fenics-packages/fenics
!apt update
!apt install -y fenicsx
import pybamm
import numpy as np
import matplotlib.pyplot as plt
print("Environment initialized.")

## Stage 1: Cell Verification
Verify the base parameter set against literature-referenced material properties.

In [ ]:
from nfpp_sodium_ion.src.calibration.verify_parameters import verify

print("Stage 1: Running Cell Verification...")
verify()

## Stage 2: Cell Optimization
Hierarchical Material Discovery + Structural Sensitivity Optimization.

In [ ]:
from src.cell_optimization.material_opt import MaterialMappingEngine
from src.cell_optimization.parameter_opts import HierarchicalOptimizer

print("Stage 2a: Running Material Property Acquisition...")
engine = MaterialMappingEngine()
material_system = engine.run()

print("\nStage 2b: Running Structural Optimization...")
optimizer = HierarchicalOptimizer(engine=engine)
optimized_res = optimizer.run()


## Stage 3: Stability Validation
Performance evaluation and resistance profile generation.

In [ ]:
from src.cell_optimization.validate import OptimizationValidator

print("Stage 3: Running Stability Validation...")

# Map optimized design vector to parameter dict
design_specs = optimized_res.get("design_specs_representative", {})
deltas = optimized_res.get("combined_deltas_representative", {})

validator = OptimizationValidator(design_specs, deltas, engine=engine)
results = validator.run_validation()

print("\nStage 3.1: Running Multi-Physics Envelope Validation...")
from src.simulation.envelope import StabilityValidator
stab_validator = StabilityValidator(base_params_updates=deltas)
envelope_res = stab_validator.validate_optimized_design({"design": [design_specs[k] for k in ["Positive electrode thickness [m]", "Negative electrode thickness [m]", "Positive electrode porosity", "Negative electrode porosity", "Positive particle radius [m]"]]})
stab_validator.export_to_json(envelope_res)

print("\nPerformance Metrics Summary:")
if results:
    for k, v in results.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

## Stage 4: Multiphysics Power Plant Model Assembly
Assemble the standalone electro-thermal and power conversion digital twin.

In [ ]:
print("Stage 4: Assembling Simscape Power Plant Model...")
matlab_build_cmd = "matlab -nodisplay -nosplash -r \"addpath('src/power_plant'); build_physical_plant(struct()); quit;\""

try:
    subprocess.run(matlab_build_cmd, shell=True, check=True)
    print("Power plant model structure generated and verified.")
except subprocess.CalledProcessError:
    print("MATLAB execution skipped (Environment check). Component files (SSC) are ready in src/power_plant/.")